# Conditional Diffusion Demo for DiffMM

Notebook nay giup ban thu nghiem phien ban Conditional Diffusion da duoc them vao du an DiffMM.

Noi dung bao gom:
- thiet lap moi truong va cau hinh chay notebook
- nap du lieu TikTok de smoke test
- tao vector dieu kien theo modality
- chay mot vong train/test ngan
- luu ket qua ra file JSON trong workspace

## 1. Thiet lap moi truong

Cell nay kiem tra Python, dat working directory ve root cua du an va cau hinh fallback khi khong co CUDA.

In [2]:
import json
import os
import platform
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != "DiffMM":
    PROJECT_ROOT = Path("/Users/cao.duc.soan/Documents/soancd/DiffMM")
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Tranh argparse cua project doc nham tham so do Jupyter truyen vao.
sys.argv = ["notebook"]

import numpy as np
import torch

if not torch.cuda.is_available():
    torch.Tensor.cuda = lambda self, device=None, non_blocking=False: self
    torch.nn.Module.cuda = lambda self, device=None: self

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Project root:", PROJECT_ROOT)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Python: 3.9.6
Platform: macOS-26.4-arm64-arm-64bit
Project root: /Users/cao.duc.soan/Documents/soancd/DiffMM
Torch: 2.8.0
CUDA available: False


## 2. Import thu vien

Nap cac module trong project can thiet cho viec smoke test Conditional Diffusion.

In [3]:
from Params import args
from DataHandler import DataHandler
from Main import Coach, seed_it
from Model import Denoise, GaussianDiffusion, Model

print("Project modules imported successfully")

Project modules imported successfully


## 3. Khai bao tham so dau vao

Dieu chinh cac gia tri nay de doi dataset, so epoch, batch size hoac bat/tat Conditional Diffusion.

In [4]:
config = {
    "data": "tiktok",
    "epoch": 1,
    "batch": 4096,
    "tstBat": 512,
    "reg": 1e-4,
    "ssl_reg": 1e-2,
    "trans": 1,
    "e_loss": 0.1,
    "cl_method": 1,
    "conditional_diffusion": True,
    "cond_dim": 64,
    "cond_dropout": 0.1,
    "seed": 421,
}

for key, value in config.items():
    setattr(args, key, value)

seed_it(args.seed)
config

{'data': 'tiktok',
 'epoch': 1,
 'batch': 4096,
 'tstBat': 512,
 'reg': 0.0001,
 'ssl_reg': 0.01,
 'trans': 1,
 'e_loss': 0.1,
 'cl_method': 1,
 'conditional_diffusion': True,
 'cond_dim': 64,
 'cond_dropout': 0.1,
 'seed': 421}

## 4. Tao hoac nap du lieu mau

Trong notebook nay, du lieu mau la du lieu that trong folder Datasets. Cell nay nap data va in thong tin co ban.

In [5]:
handler = DataHandler()
handler.LoadData()

summary = {
    "users": int(args.user),
    "items": int(args.item),
    "train_interactions": int(handler.trnLoader.dataset.__len__()),
    "image_feat_dim": int(args.image_feat_dim),
    "text_feat_dim": int(args.text_feat_dim),
}
if args.data == "tiktok":
    summary["audio_feat_dim"] = int(args.audio_feat_dim)

summary

/Users/cao.duc.soan/Documents/soancd/DiffMM/DataHandler.py:58: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:655.)
  return torch.sparse.FloatTensor(idxs, vals, shape).cuda()


{'users': 9308,
 'items': 6710,
 'train_interactions': 59541,
 'image_feat_dim': 128,
 'text_feat_dim': 768,
 'audio_feat_dim': 128}

## 5. Xu ly du lieu

Cell nay lay mot batch user-item tu diffusion loader va tao vector condition theo image modality de kiem tra hinh dang du lieu.

In [6]:
batch_item, batch_index = next(iter(handler.diffusionLoader))
batch_item = batch_item.cuda()

image_feats = handler.image_feats
condition_image = torch.mm(batch_item, image_feats) / (batch_item.sum(dim=1, keepdim=True) + 1e-8)

print("batch_item shape:", tuple(batch_item.shape))
print("condition_image shape:", tuple(condition_image.shape))
print("condition_image mean/std:", float(condition_image.mean()), float(condition_image.std()))

batch_item shape: (4096, 6710)
condition_image shape: (4096, 128)
condition_image mean/std: 2.688260555267334 0.9673773050308228


## 6. Thuc thi logic chinh

Khoi tao Coach va chay mot vong train/test ngan de xac nhan Conditional Diffusion hoat dong end-to-end.

In [7]:
coach = Coach(handler)
coach.prepareModel()
train_metrics = coach.trainEpoch()
test_metrics = coach.testEpoch()

results = {
    "config": config,
    "train_metrics": train_metrics,
    "test_metrics": test_metrics,
}
results

USER 9308 ITEM 6710
NUM OF INTERACTIONS 59541
2026-04-07 20:47:45.123903: Diffusion Step 2/2
2026-04-07 20:47:45.123909: Start to re-build UI matrix
2026-04-07 20:47:48.198556: UI matrix built!


{'config': {'data': 'tiktok',
  'epoch': 1,
  'batch': 4096,
  'tstBat': 512,
  'reg': 0.0001,
  'ssl_reg': 0.01,
  'trans': 1,
  'e_loss': 0.1,
  'cl_method': 1,
  'conditional_diffusion': True,
  'cond_dim': 64,
  'cond_dropout': 0.1,
  'seed': 421},
 'train_metrics': {'Loss': 0.4742975501077516,
  'BPR Loss': 0.4496262563126428,
  'CL loss': 0.5593205222061702,
  'Di image loss': 6007.775521269652,
  'Di text loss': 157.70919126178978,
  'Di audio loss': 29.662412015292645},
 'test_metrics': {'Recall': 0.049755301794453505,
  'NDCG': np.float64(0.02624177347645628),
  'Precision': 0.002487765089722676}}

## 7. Hien thi ket qua

In gon cac metric de de theo doi ngay trong notebook.

In [8]:
print("Train metrics")
for key, value in train_metrics.items():
    print(f"- {key}: {value:.6f}")

print("\nTest metrics")
for key, value in test_metrics.items():
    print(f"- {key}: {value:.6f}")

Train metrics
- Loss: 0.474298
- BPR Loss: 0.449626
- CL loss: 0.559321
- Di image loss: 6007.775521
- Di text loss: 157.709191
- Di audio loss: 29.662412

Test metrics
- Recall: 0.049755
- NDCG: 0.026242
- Precision: 0.002488


## 8. Luu dau ra

Ghi ket qua smoke test ra file JSON de tai su dung ngoai notebook.

In [ ]:
output_path = PROJECT_ROOT / "conditional_diffusion_smoke_test.json"
with open(output_path, "w", encoding="utf-8") as fp:
    json.dump(results, fp, indent=2)

print("Saved:", output_path)